# Demo 1: Setup and import Chinook into DuckDB

This demo sets up DuckDB in a notebook, imports the Chinook SQLite database, and creates local DuckDB tables for later demos. Run the notebook from `lectures/03/demo` so `chinook.sqlite` is in the working directory.

## Schema overview

![Chinook schema](media/chinook_schema.png)

Optional: open the full PDF diagram for zooming: `media/chinook-database-diagram.pdf`.

## Step 1: Install and connect

Install packages from `lectures/03/demo/requirements.txt` in your terminal, then run:

In [1]:
%load_ext sql
%sql duckdb:///demo_chinook.duckdb?access_mode=read_only

Connecting to 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

## Step 2: Attach the Chinook SQLite database

In [2]:
%%sql
INSTALL sqlite_scanner;
LOAD sqlite_scanner;
ATTACH 'chinook.sqlite' AS chinook (TYPE SQLITE);

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

Success


In [3]:
%%sql
SHOW TABLES;
-- This should work but doesn't :(

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

name


In [4]:
%%sql
-- We can always use the duckdb information schema
SELECT table_name FROM information_schema.tables 
WHERE table_catalog = 'chinook';

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

table_name
albums
sqlite_sequence
artists
customers
employees
genres
invoices
invoice_items
media_types
playlists


Checkpoint: you should see tables like `Albums`, `Artists`, `Tracks`, `Customers`, `Invoices`, and `Invoice_Items`.

Example output (table names will match, counts not shown yet):

| name |
| --- |
| Albums |
| Artists |
| Customers |
| Invoices |
| Invoice_Items |
| Tracks |

## Step 3: Import core tables into DuckDB

In [5]:
%%sql
-- Drop existing tables if they exist to avoid errors on re-running
DROP TABLE IF EXISTS artists;
DROP TABLE IF EXISTS albums;
DROP TABLE IF EXISTS tracks;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS invoices;
DROP TABLE IF EXISTS invoice_items;

CREATE TABLE artists AS SELECT * FROM chinook.Artists;
CREATE TABLE albums AS SELECT * FROM chinook.Albums;
CREATE TABLE tracks AS SELECT * FROM chinook.Tracks;
CREATE TABLE customers AS SELECT * FROM chinook.Customers;
CREATE TABLE invoices AS SELECT * FROM chinook.Invoices;
CREATE TABLE invoice_items AS SELECT * FROM chinook.Invoice_Items;

Running query in 'duckdb:///demo_chinook.duckdb?access_mode=read_only'

RuntimeError: (_duckdb.InvalidInputException) Invalid Input Error: Cannot execute statement of type "DROP" on database "demo_chinook" which is attached in read-only mode!
[SQL: DROP TABLE IF EXISTS artists;]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [ ]:
%%sql
SELECT 'artists' AS table_name, COUNT(*) AS rows FROM artists
UNION ALL
SELECT 'albums', COUNT(*) FROM albums
UNION ALL
SELECT 'tracks', COUNT(*) FROM tracks
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'invoices', COUNT(*) FROM invoices
UNION ALL
SELECT 'invoice_items', COUNT(*) FROM invoice_items
ORDER BY table_name;

Running query in 'duckdb:///demo_chinook.duckdb'

table_name,rows
albums,347
artists,275
customers,59
invoice_items,2240
invoices,412
tracks,3503


Checkpoint: each table should have a non-zero row count.

Example output (row counts will differ by dataset version):

| table_name | rows |
| --- | --- |
| albums | 347 |
| artists | 275 |
| customers | 59 |
| invoice_items | 2240 |
| invoices | 412 |
| tracks | 3503 |

## Step 4: Mini CSV and Parquet round-trip (DuckDB-specific)

In [ ]:
%%sql
COPY (SELECT * FROM tracks LIMIT 50)
TO 'tracks_sample.csv' (HEADER, DELIMITER ',');

COPY (SELECT * FROM tracks LIMIT 50)
TO 'tracks_sample.parquet' (FORMAT PARQUET);

Running query in 'duckdb:///demo_chinook.duckdb'

Count


In [ ]:
%%sql
SELECT * FROM read_csv_auto('tracks_sample.csv') LIMIT 5;
SELECT * FROM read_parquet('tracks_sample.parquet') LIMIT 5;

Running query in 'duckdb:///demo_chinook.duckdb'

TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman",230619,3990994,0.99
4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman",252051,4331779,0.99
5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99


Checkpoint: both queries should return a small sample of tracks, and `tracks_sample.csv` / `tracks_sample.parquet` should be created in `lectures/03/demo`.